# 10 — Visualizzazioni di explainability per la presentazione

Il notebook genera esclusivamente gli output scelti per la presentazione:

- diagnostica LIME, confronti per task e confronto degli adapter con il base;
- diagnostica SHAP, confronti per task e confronto degli adapter con il base;
- attention media, rollout, profilo attraverso i layer e confronto con il base;
- accordo tra metodi e quota di importanza attribuita agli artefatti del prompt;
- indice dei grafici e tabella riepilogativa finale.

Gli screenshot HTML di LIME, SHAP e Attention restano prodotti dal notebook **09**.


In [ ]:
# CELLA 2 — Dipendenze.

%pip install -q pandas numpy matplotlib

In [ ]:
# CELLA 3 — Import, configurazione e cartella di output.

from pathlib import Path
import json
import re
import shutil

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display


DEFAULT_PROJECT_ROOT = Path(
    r"C:\Users\fede2\Desktop\Project"
)

PROJECT_ROOT = (
    DEFAULT_PROJECT_ROOT
    if DEFAULT_PROJECT_ROOT.exists()
    else Path.cwd()
)


INPUT_DIR_CANDIDATES = [
    PROJECT_ROOT / "data" / "explainability",
    PROJECT_ROOT / "data" / "analysis" / "explainability",
    PROJECT_ROOT,
    Path.cwd(),
]


OUTPUT_DIR = (
    PROJECT_ROOT
    / "data"
    / "analysis"
    / "explainability_visualizations"
)

FIGURES_DIR = OUTPUT_DIR / "figures"

# Cancella gli output della precedente esecuzione, così la cartella
# conterrà esclusivamente i grafici previsti dalla presentazione.
RESET_OUTPUT_DIR = True

if RESET_OUTPUT_DIR and OUTPUT_DIR.exists():
    shutil.rmtree(OUTPUT_DIR)

FIGURES_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


MODEL_ORDER = [
    "base",
    "regulation",
    "positive_legends",
    "negative_legends",
    "negative_to_correction",
]


MODEL_DISPLAY = {
    "base": "Base",
    "regulation": "Regulation",
    "positive_legends": "Positive legends",
    "negative_legends": "Negative legends",
    "negative_to_correction": "Negative → correction",
}


TASK_ORDER = [
    "sst2_style",
    "paraphrase_style",
    "qqp_style",
    "mnli_style",
    "rte_style",
    "bias_detection",
    "stsb_style",
]


TASK_DISPLAY = {
    "sst2_style": "SST-2 style",
    "paraphrase_style": "Paraphrase style",
    "qqp_style": "QQP style",
    "mnli_style": "MNLI style",
    "rte_style": "RTE style",
    "bias_detection": "Bias detection",
    "stsb_style": "STS-B style",
}


TOP_K_TOKENS = 12
TOP_K_OVERLAP = 10

# "content_only" filtra marker chat, label e boilerplate.
# "raw" visualizza tutto.
TOKEN_VIEW = "content_only"

ATTENTION_TYPES_TO_PLOT = [
    "all_layers_mean",
]

PREFERRED_PROBABILITY_SOURCE = "lime"

SAVE_PNG = True
SHOW_FIGURES = True


plt.rcParams.update({
    "figure.dpi": 110,
    "savefig.dpi": 180,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.axisbelow": True,
    "font.size": 10,
})


chart_index_rows = []

print("PROJECT_ROOT:", PROJECT_ROOT)
print("OUTPUT_DIR:", OUTPUT_DIR)


In [ ]:
# CELLA 4 — Individua e carica gli output necessari del notebook 09.

EXPECTED_FILES = {
    "selected": "selected_explainability_examples.csv",
    "probabilities": "explainability_probabilities_raw.csv",
    "lime": "lime_explanations_raw.csv",
    "shap": "shap_explanations_raw.csv",
    "attention": "attention_token_scores_raw.csv",
}


def locate_input_file(expected_name):
    expected_path = Path(expected_name)
    stem = expected_path.stem
    suffix = expected_path.suffix

    for directory in INPUT_DIR_CANDIDATES:
        exact = directory / expected_name

        if exact.exists():
            return exact

    matches = []

    for directory in INPUT_DIR_CANDIDATES:
        if not directory.exists():
            continue

        matches.extend(
            directory.rglob(
                f"{stem}*{suffix}"
            )
        )

    matches = sorted(
        set(matches),
        key=lambda path: (
            len(path.name),
            len(str(path)),
        ),
    )

    if not matches:
        raise FileNotFoundError(
            f"File non trovato: {expected_name}"
        )

    return matches[0]


INPUT_PATHS = {
    key: locate_input_file(filename)
    for key, filename
    in EXPECTED_FILES.items()
}


for key, input_path in INPUT_PATHS.items():
    print(
        f"{key:16s} -> {input_path}"
    )


selected_df = pd.read_csv(
    INPUT_PATHS["selected"]
)

probabilities_df = pd.read_csv(
    INPUT_PATHS["probabilities"]
)

lime_df = pd.read_csv(
    INPUT_PATHS["lime"]
)

shap_df = pd.read_csv(
    INPUT_PATHS["shap"]
)

attention_df = pd.read_csv(
    INPUT_PATHS["attention"]
)


print("\nDimensioni:")
print("selected:", selected_df.shape)
print("probabilities:", probabilities_df.shape)
print("LIME:", lime_df.shape)
print("SHAP:", shap_df.shape)
print("attention:", attention_df.shape)


## 1. Preparazione dei dati

Viene costruita in memoria la tabella canonica delle predizioni, utilizzata esclusivamente per il riepilogo finale. Non vengono generati grafici separati di confidenza, margine o correttezza.


In [ ]:
# CELLA 7 — Funzioni comuni per token e grafici.

GENERIC_PROMPT_TERMS = {
    "system",
    "user",
    "assistant",
    "input",
    "output",
    "answer",
    "class",
    "classify",
    "classification",
    "determine",
    "evaluate",
    "following",
    "provided",
    "case",
    "scenario",
    "statement",
    "statements",
    "question",
    "questions",
    "premise",
    "hypothesis",
    "compare",
    "using",
    "use",
    "exactly",
    "one",
    "allowed",
    "label",
    "labels",
    "return",
    "only",
    "without",
    "explanation",
    "explanations",
    "punctuation",
    "additional",
    "text",
    "integer",
    "score",
    "scores",
    "relationship",
    "whether",
}


candidate_labels = set(
    probabilities_df[
        "candidate_label"
    ]
    .dropna()
    .astype(str)
    .str.lower()
)


LABEL_TERMS = set()

for label in candidate_labels:
    normalized_label = (
        label
        .replace("_", " ")
        .replace("-", " ")
    )

    LABEL_TERMS.add(normalized_label)

    LABEL_TERMS.update(
        re.findall(
            r"[a-z0-9]+",
            normalized_label,
        )
    )


PROMPT_ARTIFACT_TERMS = (
    GENERIC_PROMPT_TERMS
    | LABEL_TERMS
)


def normalize_token(value):
    if pd.isna(value):
        return ""

    text = str(value)

    for marker in [
        "Ġ",
        "▁",
        "Ċ",
        "\\n",
        "_",
    ]:
        text = text.replace(
            marker,
            " ",
        )

    text = text.strip().lower()

    text = re.sub(
        r"\s+",
        " ",
        text,
    )

    text = re.sub(
        r"^[^\w<|]+|[^\w>|]+$",
        "",
        text,
    )

    return text


def is_artifact(token):
    token = normalize_token(token)

    if not token:
        return True

    if re.fullmatch(
        r"<\|.*?\|>",
        token,
    ):
        return True

    if not re.search(
        r"[a-z0-9]",
        token,
    ):
        return True

    if token in PROMPT_ARTIFACT_TERMS:
        return True

    parts = re.findall(
        r"[a-z0-9]+",
        token,
    )

    return bool(
        parts
        and all(
            part in PROMPT_ARTIFACT_TERMS
            for part in parts
        )
    )


def prepare_tokens(
    dataframe,
    source_column,
):
    prepared = dataframe.copy()

    prepared[
        "display_token"
    ] = (
        prepared[
            source_column
        ]
        .map(normalize_token)
    )

    prepared[
        "is_prompt_artifact"
    ] = (
        prepared[
            "display_token"
        ]
        .map(is_artifact)
    )

    return prepared[
        prepared[
            "display_token"
        ]
        .ne("")
    ].copy()


def filter_view(dataframe):
    if TOKEN_VIEW == "raw":
        return dataframe.copy()

    return dataframe[
        ~dataframe[
            "is_prompt_artifact"
        ]
    ].copy()


def model_label(value):
    return MODEL_DISPLAY.get(
        value,
        value,
    )


def task_label(value):
    return TASK_DISPLAY.get(
        value,
        value,
    )




def chart_explanation(filename):
    """Guida sintetica per i 25 grafici effettivamente usati nelle slide."""

    task_name = next(
        (
            task
            for task in TASK_ORDER
            if task in filename
        ),
        None,
    )

    task_name_label = (
        task_label(task_name)
        if task_name is not None
        else "tutti i task"
    )

    lime_task_slides = {
        task_name: 3 + task_index
        for task_index, task_name in enumerate(TASK_ORDER)
    }

    shap_task_slides = {
        task_name: 11 + task_index
        for task_index, task_name in enumerate(TASK_ORDER)
    }

    attention_task_slides = {
        task_name: 19 + task_index
        for task_index, task_name in enumerate(TASK_ORDER)
    }

    if filename == "04_lime_local_fidelity.png":
        return {
            "slide_number": 1,
            "slide_title": "LIME — Fedeltà locale delle spiegazioni",
            "what_it_shows": (
                "Quanto bene il modello surrogato di LIME replica localmente "
                "il comportamento del modello originale."
            ),
            "why_it_matters": (
                "È il controllo preliminare necessario prima di interpretare "
                "i pesi assegnati ai token."
            ),
            "how_to_read": (
                "Valori più alti indicano una spiegazione locale più fedele."
            ),
            "caution": (
                "Misura la qualità della spiegazione, non l'accuratezza del modello."
            ),
        }

    if filename == "05_lime_local_approximation_error.png":
        return {
            "slide_number": 2,
            "slide_title": "LIME — Errore di approssimazione locale",
            "what_it_shows": (
                "La distanza assoluta tra la probabilità originale e quella "
                "ricostruita dal surrogato LIME."
            ),
            "why_it_matters": (
                "Completa il controllo di fedeltà: una spiegazione robusta "
                "dovrebbe avere errore ridotto."
            ),
            "how_to_read": (
                "Valori più bassi sono migliori e vanno letti insieme al local score."
            ),
            "caution": (
                "Errore basso non implica che la predizione sia corretta."
            ),
        }

    if filename.startswith("06_lime_top_features_"):
        return {
            "slide_number": lime_task_slides[task_name],
            "slide_title": f"LIME — Contributi nel task {task_name_label}",
            "what_it_shows": (
                f"I principali token LIME per {task_name_label}, confrontati "
                "nei cinque modelli mediante pannelli con scala condivisa."
            ),
            "why_it_matters": (
                "Permette di verificare se il fine-tuning conserva, amplifica "
                "o modifica i segnali locali utilizzati per la predizione."
            ),
            "how_to_read": (
                "Barre positive sostengono la label target; barre negative la contrastano."
            ),
            "caution": (
                "L'importanza è locale e riferita agli esempi selezionati."
            ),
        }

    if filename == "13_shap_reconstruction_error.png":
        return {
            "slide_number": 10,
            "slide_title": "SHAP — Qualità della ricostruzione",
            "what_it_shows": (
                "L'errore tra la probabilità target e la ricostruzione ottenuta "
                "sommando valore di base e contributi SHAP."
            ),
            "why_it_matters": (
                "Verifica che la scomposizione additiva rappresenti correttamente "
                "l'output prima di interpretare i token."
            ),
            "how_to_read": (
                "Valori prossimi a zero indicano una ricostruzione coerente."
            ),
            "caution": (
                "Non è una metrica di performance predittiva."
            ),
        }

    if filename.startswith("14_shap_top_tokens_"):
        return {
            "slide_number": shap_task_slides[task_name],
            "slide_title": f"SHAP — Contributi nel task {task_name_label}",
            "what_it_shows": (
                f"I contributi SHAP principali per {task_name_label}, confrontati "
                "nei cinque modelli con token e scala condivisi."
            ),
            "why_it_matters": (
                "Mostra in modo additivo quanto ciascun token sposta l'output "
                "rispetto al valore di base."
            ),
            "how_to_read": (
                "Valori positivi aumentano l'output target; valori negativi lo riducono."
            ),
            "caution": (
                "Il contributo dipende dall'esempio e dalla label spiegata."
            ),
        }

    if filename == "20_attention_global_overview.png":
        return {
            "slide_number": 18,
            "slide_title": "Attention — Profilo globale sui sette task",
            "what_it_shows": (
                "Il comportamento medio dell'Attention sui sette task: "
                "concentrazione nei dieci token principali e quota assegnata "
                "agli artefatti strutturali del prompt."
            ),
            "why_it_matters": (
                "Introduce la sezione Attention con un quadro complessivo, "
                "separato dai singoli test."
            ),
            "how_to_read": (
                "Concentrazione alta indica un focus più selettivo; una quota "
                "elevata sugli artefatti segnala dipendenza dal formato del prompt."
            ),
            "caution": (
                "È una media aggregata e può nascondere differenze specifiche dei task."
            ),
        }

    if filename.startswith("21_attention_all_layers_mean_"):
        return {
            "slide_number": attention_task_slides[task_name],
            "slide_title": f"Attention media — {task_name_label}",
            "what_it_shows": (
                f"I token che ricevono maggiore Attention media in {task_name_label}, "
                "confrontati nei cinque modelli."
            ),
            "why_it_matters": (
                "Consente di osservare se il fine-tuning modifica il focus interno "
                "del modello sullo stesso input."
            ),
            "how_to_read": (
                "Confrontare gli stessi token nei cinque pannelli e la distribuzione "
                "complessiva del peso."
            ),
            "caution": (
                "Attention elevata non equivale automaticamente a importanza causale."
            ),
        }

    return {
        "slide_number": None,
        "slide_title": "",
        "what_it_shows": "",
        "why_it_matters": "",
        "how_to_read": "",
        "caution": "",
    }


def save_figure(
    figure,
    filename,
    section,
    description,
):
    path = FIGURES_DIR / filename

    figure.tight_layout()

    if SAVE_PNG:
        figure.savefig(
            path,
            bbox_inches="tight",
        )

    explanation = chart_explanation(filename)

    chart_index_rows.append({
        "section": section,
        "filename": filename,
        "path": str(path),
        "description": description,
        **explanation,
    })

    if SHOW_FIGURES:
        plt.show()
    else:
        plt.close(figure)


def grouped_task_chart(
    dataframe,
    value_column,
    title,
    ylabel,
    filename,
    section,
    ylim=None,
):
    pivot = (
        dataframe
        .pivot(
            index="task_name",
            columns="model_name",
            values=value_column,
        )
        .reindex(
            index=TASK_ORDER,
            columns=MODEL_ORDER,
        )
    )

    pivot.index = [
        task_label(value)
        for value in pivot.index
    ]

    pivot.columns = [
        model_label(value)
        for value in pivot.columns
    ]

    figure = plt.figure(
        figsize=(
            13,
            6,
        )
    )

    axis = figure.add_subplot(111)

    pivot.plot(
        kind="bar",
        ax=axis,
    )

    axis.set_title(title)
    axis.set_xlabel("Task")
    axis.set_ylabel(ylabel)

    if ylim is not None:
        axis.set_ylim(*ylim)

    axis.tick_params(
        axis="x",
        rotation=25,
    )

    axis.legend(
        title="Modello",
        bbox_to_anchor=(
            1.02,
            1,
        ),
        loc="upper left",
    )

    save_figure(
        figure,
        filename,
        section,
        title,
    )


def plot_top_tokens(
    dataframe,
    task_name,
    value_column,
    ranking_column,
    title_prefix,
    filename_prefix,
    section,
    signed=True,
):
    task_data = dataframe[
        dataframe[
            "task_name"
        ]
        == task_name
    ].copy()

    task_data = filter_view(task_data)

    if task_data.empty:
        return

    ranking = (
        task_data
        .groupby(
            "display_token",
            as_index=False,
        )[
            ranking_column
        ]
        .mean()
        .sort_values(
            ranking_column,
            ascending=False,
        )
        .head(TOP_K_TOKENS)
    )

    selected_tokens = (
        ranking[
            "display_token"
        ]
        .tolist()
    )

    pivot = (
        task_data[
            task_data[
                "display_token"
            ]
            .isin(selected_tokens)
        ]
        .pivot_table(
            index="display_token",
            columns="model_name",
            values=value_column,
            aggfunc="sum",
            fill_value=0,
        )
        .reindex(
            columns=MODEL_ORDER,
            fill_value=0,
        )
    )

    ordering = (
        pivot
        .abs()
        .mean(axis=1)
        .sort_values()
        .index
        if signed
        else
        pivot
        .mean(axis=1)
        .sort_values()
        .index
    )

    pivot = pivot.loc[ordering]

    pivot.columns = [
        model_label(value)
        for value in pivot.columns
    ]

    figure = plt.figure(
        figsize=(
            13,
            max(
                6,
                0.48 * len(pivot) + 2,
            ),
        )
    )

    axis = figure.add_subplot(111)

    pivot.plot(
        kind="barh",
        ax=axis,
    )

    if signed:
        axis.axvline(
            0,
            linewidth=1,
        )

    axis.set_title(
        f"{title_prefix} — "
        f"{task_label(task_name)}"
    )

    axis.set_xlabel(
        "Contributo firmato"
        if signed
        else "Score aggregato"
    )

    axis.set_ylabel(
        "Token / feature"
    )

    axis.legend(
        title="Modello",
        bbox_to_anchor=(
            1.02,
            1,
        ),
        loc="upper left",
    )

    save_figure(
        figure,
        f"{filename_prefix}_{task_name}.png",
        section,
        (
            f"{title_prefix} per "
            f"{task_label(task_name)}."
        ),
    )

In [ ]:
# CELLA 6 — Tabella canonica delle predizioni usata nel riepilogo finale.

sources = (
    probabilities_df[
        "method_source"
    ]
    .dropna()
    .unique()
    .tolist()
)


probability_source = (
    PREFERRED_PROBABILITY_SOURCE
    if PREFERRED_PROBABILITY_SOURCE
    in sources
    else sources[0]
)


canonical_probabilities_df = probabilities_df[
    probabilities_df[
        "method_source"
    ]
    == probability_source
].copy()


prediction_rows = []


for group_key, group in (
    canonical_probabilities_df
    .groupby(
        [
            "model_name",
            "task_name",
            "example_id",
        ],
        sort=False,
    )
):
    ordered = (
        group
        .sort_values(
            "candidate_probability",
            ascending=False,
        )
        .reset_index(drop=True)
    )

    first = ordered.iloc[0]

    first_probability = float(
        first[
            "candidate_probability"
        ]
    )

    second_probability = (
        float(
            ordered.iloc[1][
                "candidate_probability"
            ]
        )
        if len(ordered) > 1
        else np.nan
    )

    gold = str(
        first[
            "gold_label"
        ]
    )

    predicted = str(
        first[
            "predicted_label"
        ]
    )

    prediction_rows.append({
        "model_name": group_key[0],
        "task_name": group_key[1],
        "example_id": group_key[2],
        "gold_label": gold,
        "predicted_label": predicted,
        "target_label": str(
            first[
                "target_label"
            ]
        ),
        "predicted_probability": first_probability,
        "second_probability": second_probability,
        "confidence_margin": (
            first_probability
            - second_probability
        ),
        "is_correct": (
            normalize_token(gold)
            == normalize_token(predicted)
        ),
    })


prediction_summary_df = pd.DataFrame(
    prediction_rows
)





## 2. LIME

Output generati:

- `04_lime_local_fidelity.png`;
- `05_lime_local_approximation_error.png`;
- sette grafici `06_lime_top_features_<task>.png`;
- `46_adapter_vs_base_lime.png`.


In [ ]:
# CELLA 12 — Diagnostica e contributi LIME con visualizzazioni differenziate.

lime_diagnostics_df = (
    lime_df[
        [
            "model_name",
            "task_name",
            "example_id",
            "target_probability",
            "lime_local_prediction",
            "lime_local_score",
            "num_samples",
        ]
    ]
    .drop_duplicates(
        subset=[
            "model_name",
            "task_name",
            "example_id",
        ]
    )
    .copy()
)


lime_diagnostics_df[
    "local_approximation_error"
] = (
    lime_diagnostics_df[
        "target_probability"
    ]
    - lime_diagnostics_df[
        "lime_local_prediction"
    ]
).abs()


_default_model_colors = (
    plt.rcParams[
        "axes.prop_cycle"
    ]
    .by_key()
    .get(
        "color",
        ["C0", "C1", "C2", "C3", "C4"],
    )
)

MODEL_COLORS = {
    model_name: _default_model_colors[
        index % len(_default_model_colors)
    ]
    for index, model_name in enumerate(MODEL_ORDER)
}

MODEL_MARKERS = {
    "base": "o",
    "regulation": "s",
    "positive_legends": "^",
    "negative_legends": "D",
    "negative_to_correction": "P",
}


# -----------------------------------------------------------------------------
# 1. Fedeltà locale LIME: range-dot plot.
#    La linea grigia mostra l'intervallo tra i modelli; i punti consentono di
#    identificare immediatamente chi approssima meglio ciascun task.
# -----------------------------------------------------------------------------

fidelity_pivot = (
    lime_diagnostics_df
    .pivot_table(
        index="task_name",
        columns="model_name",
        values="lime_local_score",
        aggfunc="mean",
    )
    .reindex(
        index=TASK_ORDER,
        columns=MODEL_ORDER,
    )
)

figure, axis = plt.subplots(
    figsize=(12.5, 7),
)

task_positions = np.arange(
    len(TASK_ORDER)
)

model_offsets = np.linspace(
    -0.20,
    0.20,
    len(MODEL_ORDER),
)

finite_fidelity = (
    fidelity_pivot
    .to_numpy(dtype=float)
)
finite_fidelity = finite_fidelity[
    np.isfinite(finite_fidelity)
]

for task_index, task_name in enumerate(TASK_ORDER):
    row = fidelity_pivot.loc[
        task_name
    ].dropna()

    if row.empty:
        continue

    axis.hlines(
        task_index,
        row.min(),
        row.max(),
        linewidth=3,
        color="0.78",
        zorder=1,
    )

    mean_value = float(
        row.mean()
    )

    axis.scatter(
        mean_value,
        task_index,
        s=170,
        facecolors="none",
        edgecolors="0.25",
        linewidths=1.3,
        zorder=2,
    )

    for model_index, model_name in enumerate(MODEL_ORDER):
        value = fidelity_pivot.loc[
            task_name,
            model_name,
        ]

        if not np.isfinite(value):
            continue

        axis.scatter(
            value,
            task_index + model_offsets[model_index],
            s=72,
            color=MODEL_COLORS[model_name],
            marker=MODEL_MARKERS[model_name],
            edgecolor="white",
            linewidth=0.6,
            label=(
                model_label(model_name)
                if task_index == 0
                else None
            ),
            zorder=3,
        )

axis.set_yticks(
    task_positions
)
axis.set_yticklabels(
    [task_label(value) for value in TASK_ORDER]
)

axis.set_xlabel(
    "LIME local score — valori più alti indicano maggiore fedeltà"
)
axis.set_ylabel("Task")
axis.set_title(
    "Fedeltà locale LIME — intervallo e posizione dei cinque modelli"
)

if finite_fidelity.size:
    minimum = float(
        finite_fidelity.min()
    )
    maximum = float(
        finite_fidelity.max()
    )
    padding = max(
        0.03,
        (maximum - minimum) * 0.12,
    )

    axis.set_xlim(
        max(0, minimum - padding),
        min(1, maximum + padding),
    )

axis.legend(
    title="Modello",
    bbox_to_anchor=(1.02, 1),
    loc="upper left",
)

save_figure(
    figure,
    "04_lime_local_fidelity.png",
    "lime",
    "Range-dot plot della fedeltà locale LIME per task e modello.",
)


# -----------------------------------------------------------------------------
# 2. Errore di approssimazione: heatmap task × modello.
#    Più chiara delle barre per individuare subito gli abbinamenti problematici.
# -----------------------------------------------------------------------------

error_pivot = (
    lime_diagnostics_df
    .pivot_table(
        index="task_name",
        columns="model_name",
        values="local_approximation_error",
        aggfunc="mean",
    )
    .reindex(
        index=TASK_ORDER,
        columns=MODEL_ORDER,
    )
)

figure, axis = plt.subplots(
    figsize=(12, 6.5),
)

matrix = error_pivot.to_numpy(
    dtype=float
)

finite_values = matrix[
    np.isfinite(matrix)
]

vmax = (
    float(finite_values.max())
    if finite_values.size
    else 1.0
)

if vmax <= 0:
    vmax = 1.0

image = axis.imshow(
    matrix,
    aspect="auto",
    cmap="YlOrRd",
    vmin=0,
    vmax=vmax,
)

axis.set_title(
    "Errore dell'approssimazione locale LIME — valori più bassi sono migliori",
    pad=16,
)
axis.set_xlabel("Modello")
axis.set_ylabel("Task")

axis.set_xticks(
    np.arange(len(MODEL_ORDER))
)
axis.set_xticklabels(
    [model_label(value) for value in MODEL_ORDER],
    rotation=20,
    ha="right",
)

axis.set_yticks(
    np.arange(len(TASK_ORDER))
)
axis.set_yticklabels(
    [task_label(value) for value in TASK_ORDER]
)

for row_index in range(matrix.shape[0]):
    for column_index in range(matrix.shape[1]):
        value = matrix[
            row_index,
            column_index,
        ]

        if not np.isfinite(value):
            label = "n.d."
        elif value < 0.001:
            label = f"{value:.1e}"
        else:
            label = f"{value:.3f}"

        normalized_value = (
            value / vmax
            if np.isfinite(value)
            else 0
        )

        axis.text(
            column_index,
            row_index,
            label,
            ha="center",
            va="center",
            fontsize=8,
            color=(
                "white"
                if normalized_value > 0.58
                else "black"
            ),
        )

colorbar = figure.colorbar(
    image,
    ax=axis,
    shrink=0.88,
    pad=0.03,
)
colorbar.set_label(
    "|probabilità modello − predizione locale LIME|"
)

axis.grid(False)

save_figure(
    figure,
    "05_lime_local_approximation_error.png",
    "lime",
    "Heatmap dell'errore di approssimazione LIME per task e modello.",
)


# -----------------------------------------------------------------------------
# 3. Preparazione dei contributi token-level.
# -----------------------------------------------------------------------------

lime_prepared_df = prepare_tokens(
    lime_df,
    "feature",
)


lime_summary_df = (
    lime_prepared_df
    .groupby(
        [
            "model_name",
            "task_name",
            "example_id",
            "display_token",
            "is_prompt_artifact",
        ],
        as_index=False,
    )
    .agg(
        lime_weight=(
            "lime_weight",
            "sum",
        ),
        absolute_lime_weight=(
            "absolute_lime_weight",
            "sum",
        ),
    )
)


def build_lime_task_pivot(
    task_name,
    top_k=10,
):
    """Costruisce una matrice token × modello sui token LIME più influenti."""

    task_data = lime_summary_df[
        lime_summary_df[
            "task_name"
        ]
        == task_name
    ].copy()

    task_data = filter_view(
        task_data
    )

    if task_data.empty:
        return pd.DataFrame()

    token_ranking = (
        task_data
        .groupby(
            "display_token",
            as_index=False,
        )[
            "absolute_lime_weight"
        ]
        .mean()
        .sort_values(
            "absolute_lime_weight",
            ascending=False,
        )
        .head(top_k)
    )

    selected_tokens = token_ranking[
        "display_token"
    ].tolist()

    pivot = (
        task_data[
            task_data[
                "display_token"
            ]
            .isin(selected_tokens)
        ]
        .pivot_table(
            index="display_token",
            columns="model_name",
            values="lime_weight",
            aggfunc="sum",
            fill_value=0,
        )
        .reindex(
            columns=MODEL_ORDER,
            fill_value=0,
        )
    )

    ordering = (
        pivot
        .abs()
        .mean(axis=1)
        .sort_values()
        .index
    )

    return pivot.loc[
        ordering
    ]


def save_lime_task_figure(
    figure,
    task_name,
    visual_description,
):
    save_figure(
        figure,
        f"06_lime_top_features_{task_name}.png",
        "lime",
        (
            f"{visual_description} per "
            f"{task_label(task_name)}."
        ),
    )


# -----------------------------------------------------------------------------
# SST-2: barre orizzontali raggruppate.
# La vista più diretta quando molti modelli condividono gli stessi token.
# -----------------------------------------------------------------------------

def plot_lime_grouped_bars(
    task_name,
):
    pivot = build_lime_task_pivot(
        task_name,
        top_k=9,
    )

    if pivot.empty:
        return

    figure, axis = plt.subplots(
        figsize=(13, 7),
    )

    token_positions = np.arange(
        len(pivot)
    )

    bar_height = 0.14
    offsets = (
        np.arange(len(MODEL_ORDER))
        - (len(MODEL_ORDER) - 1) / 2
    ) * bar_height

    for model_index, model_name in enumerate(MODEL_ORDER):
        axis.barh(
            token_positions + offsets[model_index],
            pivot[
                model_name
            ].to_numpy(),
            height=bar_height * 0.9,
            label=model_label(model_name),
            color=MODEL_COLORS[model_name],
        )

    axis.axvline(
        0,
        linewidth=1.1,
        color="black",
    )

    axis.set_yticks(
        token_positions
    )
    axis.set_yticklabels(
        pivot.index
    )
    axis.set_xlabel(
        "Peso LIME firmato"
    )
    axis.set_ylabel("Token")
    axis.set_title(
        f"Contributi LIME condivisi — {task_label(task_name)}"
    )

    axis.legend(
        title="Modello",
        bbox_to_anchor=(1.02, 1),
        loc="upper left",
    )

    save_lime_task_figure(
        figure,
        task_name,
        "Barre raggruppate sui token condivisi",
    )


# -----------------------------------------------------------------------------
# Paraphrase: heatmap divergente annotata.
# Fa emergere subito inversioni di segno e differenze di intensità.
# -----------------------------------------------------------------------------

def plot_lime_heatmap(
    task_name,
):
    pivot = build_lime_task_pivot(
        task_name,
        top_k=10,
    )

    if pivot.empty:
        return

    pivot = pivot.iloc[
        ::-1
    ]

    matrix = pivot.to_numpy(
        dtype=float
    )

    max_absolute = float(
        np.nanmax(
            np.abs(matrix)
        )
    )

    if max_absolute <= 0:
        max_absolute = 1.0

    figure, axis = plt.subplots(
        figsize=(11.5, 7),
    )

    image = axis.imshow(
        matrix,
        aspect="auto",
        cmap="RdBu_r",
        vmin=-max_absolute,
        vmax=max_absolute,
    )

    axis.set_xticks(
        np.arange(len(MODEL_ORDER))
    )
    axis.set_xticklabels(
        [model_label(value) for value in MODEL_ORDER],
        rotation=20,
        ha="right",
    )

    axis.set_yticks(
        np.arange(len(pivot))
    )
    axis.set_yticklabels(
        pivot.index
    )

    axis.set_xlabel("Modello")
    axis.set_ylabel("Token")
    axis.set_title(
        f"Mappa dei contributi LIME — {task_label(task_name)}"
    )

    for row_index in range(matrix.shape[0]):
        for column_index in range(matrix.shape[1]):
            value = matrix[
                row_index,
                column_index,
            ]

            axis.text(
                column_index,
                row_index,
                f"{value:+.3f}",
                ha="center",
                va="center",
                fontsize=7.5,
                color=(
                    "white"
                    if abs(value) > 0.58 * max_absolute
                    else "black"
                ),
            )

    colorbar = figure.colorbar(
        image,
        ax=axis,
        shrink=0.88,
        pad=0.03,
    )
    colorbar.set_label(
        "Peso LIME firmato"
    )

    axis.grid(False)

    save_lime_task_figure(
        figure,
        task_name,
        "Heatmap divergente token per modello",
    )


# -----------------------------------------------------------------------------
# QQP: lollipop plot.
# Il punto è il peso; il segmento rende evidente distanza e direzione dallo zero.
# -----------------------------------------------------------------------------

def plot_lime_lollipop(
    task_name,
):
    pivot = build_lime_task_pivot(
        task_name,
        top_k=10,
    )

    if pivot.empty:
        return

    figure, axis = plt.subplots(
        figsize=(13, 7.5),
    )

    base_positions = np.arange(
        len(pivot)
    )

    offsets = np.linspace(
        -0.24,
        0.24,
        len(MODEL_ORDER),
    )

    for model_index, model_name in enumerate(MODEL_ORDER):
        values = pivot[
            model_name
        ].to_numpy()

        y_positions = (
            base_positions
            + offsets[model_index]
        )

        axis.hlines(
            y_positions,
            0,
            values,
            color=MODEL_COLORS[model_name],
            linewidth=1.1,
            alpha=0.45,
        )

        axis.scatter(
            values,
            y_positions,
            s=56,
            color=MODEL_COLORS[model_name],
            marker=MODEL_MARKERS[model_name],
            label=model_label(model_name),
            zorder=3,
        )

    axis.axvline(
        0,
        linewidth=1.1,
        color="black",
    )

    axis.set_yticks(
        base_positions
    )
    axis.set_yticklabels(
        pivot.index
    )
    axis.set_xlabel(
        "Peso LIME firmato"
    )
    axis.set_ylabel("Token")
    axis.set_title(
        f"Lollipop dei contributi LIME — {task_label(task_name)}"
    )

    axis.legend(
        title="Modello",
        bbox_to_anchor=(1.02, 1),
        loc="upper left",
    )

    save_lime_task_figure(
        figure,
        task_name,
        "Lollipop plot dei contributi firmati",
    )


# -----------------------------------------------------------------------------
# MNLI: small multiples, un pannello per modello.
# La stessa scala e lo stesso ordine dei token evitano sovrapposizioni.
# -----------------------------------------------------------------------------

def plot_lime_small_multiples(
    task_name,
    top_k=10,
):
    """
    Genera cinque pannelli affiancati, uno per modello, usando:
    - gli stessi token;
    - lo stesso ordine dei token;
    - la stessa scala dell'asse X;
    - barre tratteggiate e attenuate per i contributi negativi.
    """

    pivot = build_lime_task_pivot(
        task_name,
        top_k=top_k,
    )

    if pivot.empty:
        return

    max_absolute = float(
        pivot
        .abs()
        .to_numpy()
        .max()
    )

    if not np.isfinite(max_absolute) or max_absolute <= 0:
        max_absolute = 1.0

    x_limit = max_absolute * 1.18

    figure_height = max(
        6.5,
        0.58 * len(pivot) + 2.2,
    )

    figure, axes = plt.subplots(
        1,
        len(MODEL_ORDER),
        figsize=(19, figure_height),
        sharex=True,
        sharey=True,
        constrained_layout=True,
    )

    axes = np.atleast_1d(
        axes
    )

    token_positions = np.arange(
        len(pivot)
    )

    for model_index, model_name in enumerate(MODEL_ORDER):
        axis = axes[
            model_index
        ]

        values = pivot[
            model_name
        ].to_numpy(
            dtype=float
        )

        bars = axis.barh(
            token_positions,
            values,
            height=0.78,
            color=MODEL_COLORS[model_name],
            edgecolor=MODEL_COLORS[model_name],
            linewidth=0.7,
        )

        for bar, value in zip(
            bars,
            values,
        ):
            if value < 0:
                bar.set_alpha(0.38)
                bar.set_hatch("//")

            if abs(value) < max_absolute * 0.025:
                continue

            horizontal_alignment = (
                "left"
                if value >= 0
                else "right"
            )

            text_offset = (
                max_absolute * 0.025
                if value >= 0
                else -max_absolute * 0.025
            )

            axis.text(
                value + text_offset,
                bar.get_y() + bar.get_height() / 2,
                f"{value:+.2f}",
                va="center",
                ha=horizontal_alignment,
                fontsize=7.5,
                color="0.25",
            )

        axis.axvline(
            0,
            linewidth=1.15,
            color="black",
            zorder=3,
        )

        axis.set_xlim(
            -x_limit,
            x_limit,
        )

        axis.set_title(
            model_label(model_name),
            fontsize=11,
            pad=10,
        )

        axis.set_xlabel(
            "Peso LIME"
        )

        axis.grid(
            axis="x",
            alpha=0.25,
        )

        if model_index == 0:
            axis.set_yticks(
                token_positions
            )
            axis.set_yticklabels(
                pivot.index
            )
            axis.set_ylabel(
                "Token"
            )
        else:
            axis.tick_params(
                axis="y",
                labelleft=False,
            )

    figure.suptitle(
        f"Contributi LIME per modello — {task_label(task_name)}",
        fontsize=15,
        y=1.03,
    )

    figure.text(
        0.5,
        -0.015,
        (
            "Barre piene: contributo positivo verso la label target · "
            "Barre tratteggiate: contributo negativo"
        ),
        ha="center",
        fontsize=9,
        color="0.35",
    )

    save_lime_task_figure(
        figure,
        task_name,
        "Small multiples con un pannello per modello e scala condivisa",
    )

# -----------------------------------------------------------------------------
# RTE: range plot.
# La linea mostra la dispersione min-max; i punti identificano i modelli.
# -----------------------------------------------------------------------------

def plot_lime_range(
    task_name,
):
    pivot = build_lime_task_pivot(
        task_name,
        top_k=10,
    )

    if pivot.empty:
        return

    figure, axis = plt.subplots(
        figsize=(13, 7.5),
    )

    token_positions = np.arange(
        len(pivot)
    )

    for token_index, token_name in enumerate(pivot.index):
        row = pivot.loc[
            token_name
        ]

        axis.hlines(
            token_index,
            row.min(),
            row.max(),
            linewidth=3,
            color="0.75",
            zorder=1,
        )

        for model_name in MODEL_ORDER:
            axis.scatter(
                row[
                    model_name
                ],
                token_index,
                s=66,
                color=MODEL_COLORS[model_name],
                marker=MODEL_MARKERS[model_name],
                edgecolor="white",
                linewidth=0.6,
                zorder=3,
                label=(
                    model_label(model_name)
                    if token_index == 0
                    else None
                ),
            )

    axis.axvline(
        0,
        linewidth=1.1,
        color="black",
    )

    axis.set_yticks(
        token_positions
    )
    axis.set_yticklabels(
        pivot.index
    )
    axis.set_xlabel(
        "Peso LIME firmato"
    )
    axis.set_ylabel("Token")
    axis.set_title(
        f"Intervallo e dispersione dei contributi LIME — {task_label(task_name)}"
    )

    axis.legend(
        title="Modello",
        bbox_to_anchor=(1.02, 1),
        loc="upper left",
    )

    save_lime_task_figure(
        figure,
        task_name,
        "Range plot con intervallo min-max e punti dei modelli",
    )


# -----------------------------------------------------------------------------
# Bias detection: bubble matrix.
# Dimensione = importanza assoluta; colore = segno del contributo.
# -----------------------------------------------------------------------------

def plot_lime_bubble_matrix(
    task_name,
):
    pivot = build_lime_task_pivot(
        task_name,
        top_k=10,
    )

    if pivot.empty:
        return

    pivot = pivot.iloc[
        ::-1
    ]

    matrix = pivot.to_numpy(
        dtype=float
    )

    max_absolute = float(
        np.nanmax(
            np.abs(matrix)
        )
    )

    if max_absolute <= 0:
        max_absolute = 1.0

    x_grid, y_grid = np.meshgrid(
        np.arange(len(MODEL_ORDER)),
        np.arange(len(pivot)),
    )

    bubble_sizes = (
        45
        + 760
        * np.abs(matrix)
        / max_absolute
    )

    figure, axis = plt.subplots(
        figsize=(11.5, 7.5),
    )

    scatter = axis.scatter(
        x_grid.ravel(),
        y_grid.ravel(),
        s=bubble_sizes.ravel(),
        c=matrix.ravel(),
        cmap="RdBu_r",
        vmin=-max_absolute,
        vmax=max_absolute,
        edgecolors="0.25",
        linewidths=0.5,
        alpha=0.88,
    )

    axis.set_xticks(
        np.arange(len(MODEL_ORDER))
    )
    axis.set_xticklabels(
        [model_label(value) for value in MODEL_ORDER],
        rotation=20,
        ha="right",
    )

    axis.set_yticks(
        np.arange(len(pivot))
    )
    axis.set_yticklabels(
        pivot.index
    )

    axis.set_xlabel("Modello")
    axis.set_ylabel("Token")
    axis.set_title(
        f"Segno e intensità dei contributi LIME — {task_label(task_name)}"
    )

    for row_index in range(matrix.shape[0]):
        for column_index in range(matrix.shape[1]):
            value = matrix[
                row_index,
                column_index,
            ]

            if abs(value) >= 0.45 * max_absolute:
                axis.text(
                    column_index,
                    row_index,
                    f"{value:+.2f}",
                    ha="center",
                    va="center",
                    fontsize=7,
                )

    colorbar = figure.colorbar(
        scatter,
        ax=axis,
        shrink=0.88,
        pad=0.03,
    )
    colorbar.set_label(
        "Peso LIME firmato"
    )

    size_handles = []
    size_labels = []

    for fraction, label in [
        (0.25, "Basso"),
        (0.55, "Medio"),
        (1.00, "Alto"),
    ]:
        size_handles.append(
            axis.scatter(
                [],
                [],
                s=45 + 760 * fraction,
                color="0.78",
                edgecolor="0.25",
            )
        )
        size_labels.append(
            label
        )

    axis.legend(
        size_handles,
        size_labels,
        title="Importanza assoluta",
        bbox_to_anchor=(1.18, 0.42),
        loc="center left",
        frameon=True,
    )

    axis.grid(False)

    save_lime_task_figure(
        figure,
        task_name,
        "Bubble matrix con segno e intensità dei contributi",
    )


# -----------------------------------------------------------------------------
# STS-B: contribution swarm.
# Ogni riga è un modello; posizione e dimensione mostrano direzione e intensità.
# -----------------------------------------------------------------------------

def plot_lime_contribution_swarm(
    task_name,
):
    task_data = lime_summary_df[
        lime_summary_df[
            "task_name"
        ]
        == task_name
    ].copy()

    task_data = filter_view(
        task_data
    )

    if task_data.empty:
        return

    figure, axis = plt.subplots(
        figsize=(13, 7),
    )

    plotted_values = []

    for model_index, model_name in enumerate(MODEL_ORDER):
        model_data = (
            task_data[
                task_data[
                    "model_name"
                ]
                == model_name
            ]
            .groupby(
                "display_token",
                as_index=False,
            )
            .agg(
                lime_weight=(
                    "lime_weight",
                    "sum",
                ),
                absolute_lime_weight=(
                    "absolute_lime_weight",
                    "sum",
                ),
            )
            .sort_values(
                "absolute_lime_weight",
                ascending=False,
            )
            .head(8)
            .reset_index(
                drop=True
            )
        )

        if model_data.empty:
            continue

        values = model_data[
            "lime_weight"
        ].to_numpy()

        plotted_values.extend(
            values.tolist()
        )

        offsets = np.linspace(
            -0.22,
            0.22,
            len(model_data),
        )

        y_positions = (
            model_index
            + offsets
        )

        max_model_importance = float(
            model_data[
                "absolute_lime_weight"
            ].max()
        )

        if max_model_importance <= 0:
            max_model_importance = 1.0

        sizes = (
            55
            + 420
            * model_data[
                "absolute_lime_weight"
            ].to_numpy()
            / max_model_importance
        )

        axis.scatter(
            values,
            y_positions,
            s=sizes,
            color=MODEL_COLORS[model_name],
            alpha=0.78,
            edgecolor="white",
            linewidth=0.6,
        )

        for row_index in range(
            min(
                3,
                len(model_data),
            )
        ):
            row = model_data.iloc[
                row_index
            ]

            axis.annotate(
                row[
                    "display_token"
                ],
                (
                    row[
                        "lime_weight"
                    ],
                    y_positions[
                        row_index
                    ],
                ),
                xytext=(5, 4),
                textcoords="offset points",
                fontsize=8,
            )

    axis.axvline(
        0,
        linewidth=1.1,
        color="black",
    )

    axis.set_yticks(
        np.arange(len(MODEL_ORDER))
    )
    axis.set_yticklabels(
        [model_label(value) for value in MODEL_ORDER]
    )

    axis.set_xlabel(
        "Peso LIME firmato"
    )
    axis.set_ylabel("Modello")
    axis.set_title(
        f"Distribuzione dei principali contributi LIME — {task_label(task_name)}"
    )

    if plotted_values:
        max_absolute = max(
            abs(value)
            for value in plotted_values
        )

        axis.set_xlim(
            -max_absolute * 1.28,
            max_absolute * 1.28,
        )

    save_lime_task_figure(
        figure,
        task_name,
        "Contribution swarm con annotazione dei token più forti",
    )


# -----------------------------------------------------------------------------
# Aggiorna la guida di lettura salvata nel chart_index.csv.
# -----------------------------------------------------------------------------

if "_chart_explanation_before_lime_redesign" not in globals():
    _chart_explanation_before_lime_redesign = chart_explanation


def chart_explanation(
    filename,
):
    explanation = _chart_explanation_before_lime_redesign(
        filename
    )

    if filename == "04_lime_local_fidelity.png":
        explanation[
            "how_to_read"
        ] = (
            "Ogni riga è un task. La linea grigia mostra la distanza tra il modello "
            "con fedeltà più bassa e quello con fedeltà più alta; i simboli identificano "
            "i cinque modelli e il cerchio vuoto rappresenta la media del task."
        )

    if filename == "05_lime_local_approximation_error.png":
        explanation[
            "how_to_read"
        ] = (
            "Leggere la matrice per riga: ogni riga è un task e ogni colonna un modello. "
            "Celle chiare e valori vicini a zero indicano una ricostruzione locale più "
            "precisa; celle più scure segnalano i casi da trattare con maggiore cautela."
        )

    visual_guides = {
        task_name: (
            "Small multiples: ogni pannello rappresenta uno dei cinque modelli. "
            "I token sono allineati nello stesso ordine e la scala X è condivisa; "
            "le barre piene sostengono la label target, quelle tratteggiate la contrastano."
        )
        for task_name in TASK_ORDER
    }

    if filename.startswith(
        "06_lime_top_features_"
    ):
        for task_name, guide in visual_guides.items():
            if task_name in filename:
                explanation[
                    "how_to_read"
                ] = guide
                break

    return explanation


# Le diagnostiche sono state salvate prima della ridefinizione della guida.
# Aggiorna quindi le relative righe già presenti nell'indice.
for _chart_row in chart_index_rows:
    if _chart_row.get("filename") in {
        "04_lime_local_fidelity.png",
        "05_lime_local_approximation_error.png",
    }:
        _chart_row.update(
            chart_explanation(
                _chart_row[
                    "filename"
                ]
            )
        )


# -----------------------------------------------------------------------------
# Generazione uniforme dei contributi LIME.
# Tutti i task usano small multiples: un pannello per ciascuno dei cinque
# modelli, token allineati e scala X condivisa.
# I nomi dei file restano invariati.
# -----------------------------------------------------------------------------

for task_name in TASK_ORDER:
    plot_lime_small_multiples(
        task_name,
        top_k=10,
    )


## 3. SHAP

Output generati:

- `13_shap_reconstruction_error.png`;
- sette grafici `14_shap_top_tokens_<task>.png`;
- `46_adapter_vs_base_shap.png`.


In [ ]:
# CELLA 14 — Diagnostica e contributi SHAP con visualizzazioni differenziate.

shap_diagnostics_df = (
    shap_df
    .groupby(
        [
            "model_name",
            "task_name",
            "example_id",
        ],
        as_index=False,
    )
    .agg(
        target_probability=(
            "target_probability",
            "first",
        ),
        base_value=(
            "base_value",
            "first",
        ),
        shap_sum=(
            "shap_value",
            "sum",
        ),
        token_count=(
            "token_index",
            "count",
        ),
    )
)


shap_diagnostics_df[
    "reconstructed_probability"
] = (
    shap_diagnostics_df[
        "base_value"
    ]
    + shap_diagnostics_df[
        "shap_sum"
    ]
)


shap_diagnostics_df[
    "reconstruction_error"
] = (
    shap_diagnostics_df[
        "target_probability"
    ]
    - shap_diagnostics_df[
        "reconstructed_probability"
    ]
).abs()


# -----------------------------------------------------------------------------
# 1. Diagnostica SHAP: heatmap task × modello.
#    Più leggibile del precedente grafico a barre perché rende immediatamente
#    visibili le celle con errore elevato. Colori chiari = errore basso.
# -----------------------------------------------------------------------------

diagnostic_pivot = (
    shap_diagnostics_df
    .pivot_table(
        index="task_name",
        columns="model_name",
        values="reconstruction_error",
        aggfunc="mean",
    )
    .reindex(
        index=TASK_ORDER,
        columns=MODEL_ORDER,
    )
)

figure, axis = plt.subplots(
    figsize=(12, 6.5),
)

matrix = diagnostic_pivot.to_numpy(dtype=float)
finite_values = matrix[np.isfinite(matrix)]

vmax = (
    float(finite_values.max())
    if finite_values.size
    else 1.0
)

if vmax <= 0:
    vmax = 1.0

image = axis.imshow(
    matrix,
    aspect="auto",
    cmap="YlOrRd",
    vmin=0,
    vmax=vmax,
)

axis.set_title(
    "Errore di ricostruzione SHAP — valori più bassi sono migliori",
    pad=16,
)
axis.set_xlabel("Modello")
axis.set_ylabel("Task")

axis.set_xticks(
    np.arange(len(MODEL_ORDER))
)
axis.set_xticklabels(
    [model_label(value) for value in MODEL_ORDER],
    rotation=20,
    ha="right",
)

axis.set_yticks(
    np.arange(len(TASK_ORDER))
)
axis.set_yticklabels(
    [task_label(value) for value in TASK_ORDER]
)

for row_index in range(matrix.shape[0]):
    for column_index in range(matrix.shape[1]):
        value = matrix[
            row_index,
            column_index,
        ]

        if not np.isfinite(value):
            label = "n.d."
        elif value < 0.001:
            label = f"{value:.1e}"
        else:
            label = f"{value:.3f}"

        normalized_value = value / vmax if np.isfinite(value) else 0

        axis.text(
            column_index,
            row_index,
            label,
            ha="center",
            va="center",
            fontsize=8,
            color=(
                "white"
                if normalized_value > 0.58
                else "black"
            ),
        )

colorbar = figure.colorbar(
    image,
    ax=axis,
    shrink=0.88,
    pad=0.03,
)
colorbar.set_label(
    "|target − (base value + somma SHAP)|"
)

axis.grid(False)

save_figure(
    figure,
    "13_shap_reconstruction_error.png",
    "shap",
    "Heatmap dell'errore di ricostruzione SHAP per task e modello.",
)


# -----------------------------------------------------------------------------
# 2. Preparazione dei contributi token-level.
# -----------------------------------------------------------------------------

shap_prepared_df = prepare_tokens(
    shap_df,
    "token",
)


shap_summary_df = (
    shap_prepared_df
    .groupby(
        [
            "model_name",
            "task_name",
            "example_id",
            "display_token",
            "is_prompt_artifact",
        ],
        as_index=False,
    )
    .agg(
        shap_value=(
            "shap_value",
            "sum",
        ),
        absolute_shap_value=(
            "absolute_shap_value",
            "sum",
        ),
    )
)


_default_model_colors = (
    plt.rcParams[
        "axes.prop_cycle"
    ]
    .by_key()
    .get(
        "color",
        ["C0", "C1", "C2", "C3", "C4"],
    )
)

MODEL_COLORS = {
    model_name: _default_model_colors[
        index % len(_default_model_colors)
    ]
    for index, model_name in enumerate(MODEL_ORDER)
}

MODEL_MARKERS = {
    "base": "o",
    "regulation": "s",
    "positive_legends": "^",
    "negative_legends": "D",
    "negative_to_correction": "P",
}


def build_shap_task_pivot(
    task_name,
    top_k=10,
):
    """Costruisce una matrice token × modello sui token più influenti."""

    task_data = shap_summary_df[
        shap_summary_df[
            "task_name"
        ]
        == task_name
    ].copy()

    task_data = filter_view(task_data)

    if task_data.empty:
        return pd.DataFrame()

    token_ranking = (
        task_data
        .groupby(
            "display_token",
            as_index=False,
        )[
            "absolute_shap_value"
        ]
        .mean()
        .sort_values(
            "absolute_shap_value",
            ascending=False,
        )
        .head(top_k)
    )

    selected_tokens = token_ranking[
        "display_token"
    ].tolist()

    pivot = (
        task_data[
            task_data[
                "display_token"
            ]
            .isin(selected_tokens)
        ]
        .pivot_table(
            index="display_token",
            columns="model_name",
            values="shap_value",
            aggfunc="sum",
            fill_value=0,
        )
        .reindex(
            columns=MODEL_ORDER,
            fill_value=0,
        )
    )

    ordering = (
        pivot
        .abs()
        .mean(axis=1)
        .sort_values()
        .index
    )

    return pivot.loc[ordering]


def save_shap_task_figure(
    figure,
    task_name,
    visual_description,
):
    save_figure(
        figure,
        f"14_shap_top_tokens_{task_name}.png",
        "shap",
        (
            f"{visual_description} per "
            f"{task_label(task_name)}."
        ),
    )


# -----------------------------------------------------------------------------
# SST-2: barre orizzontali raggruppate.
# Ottime quando i modelli utilizzano in larga parte gli stessi token.
# -----------------------------------------------------------------------------

def plot_shap_grouped_bars(task_name):
    pivot = build_shap_task_pivot(
        task_name,
        top_k=9,
    )

    if pivot.empty:
        return

    figure, axis = plt.subplots(
        figsize=(13, 7),
    )

    token_positions = np.arange(
        len(pivot)
    )

    bar_height = 0.14
    offsets = (
        np.arange(len(MODEL_ORDER))
        - (len(MODEL_ORDER) - 1) / 2
    ) * bar_height

    for model_index, model_name in enumerate(MODEL_ORDER):
        axis.barh(
            token_positions + offsets[model_index],
            pivot[model_name].to_numpy(),
            height=bar_height * 0.9,
            label=model_label(model_name),
            color=MODEL_COLORS[model_name],
        )

    axis.axvline(
        0,
        linewidth=1.1,
        color="black",
    )

    axis.set_yticks(token_positions)
    axis.set_yticklabels(pivot.index)
    axis.set_xlabel("Contributo SHAP firmato")
    axis.set_ylabel("Token")
    axis.set_title(
        f"Contributi SHAP condivisi — {task_label(task_name)}"
    )

    axis.legend(
        title="Modello",
        bbox_to_anchor=(1.02, 1),
        loc="upper left",
    )

    save_shap_task_figure(
        figure,
        task_name,
        "Barre raggruppate sui token condivisi",
    )


# -----------------------------------------------------------------------------
# Paraphrase: lollipop plot.
# Mantiene leggibili segno, intensità e differenze tra modelli senza masse di barre.
# -----------------------------------------------------------------------------

def plot_shap_lollipop(task_name):
    pivot = build_shap_task_pivot(
        task_name,
        top_k=10,
    )

    if pivot.empty:
        return

    figure, axis = plt.subplots(
        figsize=(13, 7.5),
    )

    base_positions = np.arange(
        len(pivot)
    )
    offsets = np.linspace(
        -0.24,
        0.24,
        len(MODEL_ORDER),
    )

    for model_index, model_name in enumerate(MODEL_ORDER):
        values = pivot[
            model_name
        ].to_numpy()

        y_positions = (
            base_positions
            + offsets[model_index]
        )

        axis.hlines(
            y_positions,
            0,
            values,
            color=MODEL_COLORS[model_name],
            linewidth=1.1,
            alpha=0.45,
        )

        axis.scatter(
            values,
            y_positions,
            s=56,
            color=MODEL_COLORS[model_name],
            marker=MODEL_MARKERS[model_name],
            label=model_label(model_name),
            zorder=3,
        )

    axis.axvline(
        0,
        linewidth=1.1,
        color="black",
    )

    axis.set_yticks(base_positions)
    axis.set_yticklabels(pivot.index)
    axis.set_xlabel("Contributo SHAP firmato")
    axis.set_ylabel("Token")
    axis.set_title(
        f"Lollipop dei contributi SHAP — {task_label(task_name)}"
    )

    axis.legend(
        title="Modello",
        bbox_to_anchor=(1.02, 1),
        loc="upper left",
    )

    save_shap_task_figure(
        figure,
        task_name,
        "Lollipop plot dei contributi firmati",
    )


# -----------------------------------------------------------------------------
# QQP: heatmap divergente annotata.
# È la vista più immediata per identificare inversioni di segno tra modelli.
# -----------------------------------------------------------------------------

def plot_shap_heatmap(task_name):
    pivot = build_shap_task_pivot(
        task_name,
        top_k=10,
    )

    if pivot.empty:
        return

    pivot = pivot.iloc[::-1]
    matrix = pivot.to_numpy(dtype=float)

    max_absolute = float(
        np.nanmax(np.abs(matrix))
    )

    if max_absolute <= 0:
        max_absolute = 1.0

    figure, axis = plt.subplots(
        figsize=(11.5, 7),
    )

    image = axis.imshow(
        matrix,
        aspect="auto",
        cmap="RdBu_r",
        vmin=-max_absolute,
        vmax=max_absolute,
    )

    axis.set_xticks(
        np.arange(len(MODEL_ORDER))
    )
    axis.set_xticklabels(
        [model_label(value) for value in MODEL_ORDER],
        rotation=20,
        ha="right",
    )

    axis.set_yticks(
        np.arange(len(pivot))
    )
    axis.set_yticklabels(pivot.index)

    axis.set_xlabel("Modello")
    axis.set_ylabel("Token")
    axis.set_title(
        f"Mappa dei contributi SHAP — {task_label(task_name)}"
    )

    for row_index in range(matrix.shape[0]):
        for column_index in range(matrix.shape[1]):
            value = matrix[
                row_index,
                column_index,
            ]

            axis.text(
                column_index,
                row_index,
                f"{value:+.3f}",
                ha="center",
                va="center",
                fontsize=7.5,
                color=(
                    "white"
                    if abs(value) > 0.58 * max_absolute
                    else "black"
                ),
            )

    colorbar = figure.colorbar(
        image,
        ax=axis,
        shrink=0.88,
        pad=0.03,
    )
    colorbar.set_label(
        "Contributo SHAP firmato"
    )

    axis.grid(False)

    save_shap_task_figure(
        figure,
        task_name,
        "Heatmap divergente token per modello",
    )


# -----------------------------------------------------------------------------
# MNLI: small multiples, un pannello per modello.
# Evita la sovrapposizione quando molti contributi sono vicini tra loro.
# -----------------------------------------------------------------------------

def plot_shap_small_multiples(task_name):
    pivot = build_shap_task_pivot(
        task_name,
        top_k=8,
    )

    if pivot.empty:
        return

    max_absolute = float(
        pivot.abs().to_numpy().max()
    )

    if max_absolute <= 0:
        max_absolute = 1.0

    figure, axes = plt.subplots(
        1,
        len(MODEL_ORDER),
        figsize=(18, 7),
        sharey=True,
    )

    token_positions = np.arange(
        len(pivot)
    )

    for model_index, model_name in enumerate(MODEL_ORDER):
        axis = axes[model_index]
        values = pivot[
            model_name
        ].to_numpy()

        bars = axis.barh(
            token_positions,
            values,
            color=MODEL_COLORS[model_name],
        )

        for bar, value in zip(bars, values):
            if value < 0:
                bar.set_alpha(0.38)
                bar.set_hatch("//")

        axis.axvline(
            0,
            linewidth=1,
            color="black",
        )

        axis.set_xlim(
            -max_absolute * 1.12,
            max_absolute * 1.12,
        )

        axis.set_title(
            model_label(model_name),
            fontsize=10,
        )
        axis.set_xlabel("SHAP")

        if model_index == 0:
            axis.set_yticks(token_positions)
            axis.set_yticklabels(pivot.index)
            axis.set_ylabel("Token")

    figure.suptitle(
        f"Contributi SHAP per modello — {task_label(task_name)}",
        y=1.02,
    )

    save_shap_task_figure(
        figure,
        task_name,
        "Small multiples con un pannello per modello",
    )


# -----------------------------------------------------------------------------
# RTE: range plot.
# La linea mostra l'intervallo min-max; i punti mostrano i singoli modelli.
# È molto efficace per evidenziare divergenza e inversioni.
# -----------------------------------------------------------------------------

def plot_shap_range(task_name):
    pivot = build_shap_task_pivot(
        task_name,
        top_k=10,
    )

    if pivot.empty:
        return

    figure, axis = plt.subplots(
        figsize=(13, 7.5),
    )

    token_positions = np.arange(
        len(pivot)
    )

    for token_index, token_name in enumerate(pivot.index):
        row = pivot.loc[token_name]

        axis.hlines(
            token_index,
            row.min(),
            row.max(),
            linewidth=3,
            color="0.75",
            zorder=1,
        )

        for model_name in MODEL_ORDER:
            axis.scatter(
                row[model_name],
                token_index,
                s=66,
                color=MODEL_COLORS[model_name],
                marker=MODEL_MARKERS[model_name],
                edgecolor="white",
                linewidth=0.6,
                zorder=3,
                label=(
                    model_label(model_name)
                    if token_index == 0
                    else None
                ),
            )

    axis.axvline(
        0,
        linewidth=1.1,
        color="black",
    )

    axis.set_yticks(token_positions)
    axis.set_yticklabels(pivot.index)
    axis.set_xlabel("Contributo SHAP firmato")
    axis.set_ylabel("Token")
    axis.set_title(
        f"Intervallo e dispersione dei contributi SHAP — {task_label(task_name)}"
    )

    axis.legend(
        title="Modello",
        bbox_to_anchor=(1.02, 1),
        loc="upper left",
    )

    save_shap_task_figure(
        figure,
        task_name,
        "Range plot con intervallo min-max e punti dei modelli",
    )


# -----------------------------------------------------------------------------
# Bias detection: bubble matrix.
# Colore = segno; dimensione = intensità assoluta.
# -----------------------------------------------------------------------------

def plot_shap_bubble_matrix(task_name):
    pivot = build_shap_task_pivot(
        task_name,
        top_k=10,
    )

    if pivot.empty:
        return

    pivot = pivot.iloc[::-1]
    matrix = pivot.to_numpy(dtype=float)

    max_absolute = float(
        np.nanmax(np.abs(matrix))
    )

    if max_absolute <= 0:
        max_absolute = 1.0

    x_grid, y_grid = np.meshgrid(
        np.arange(len(MODEL_ORDER)),
        np.arange(len(pivot)),
    )

    bubble_sizes = (
        45
        + 760
        * np.abs(matrix)
        / max_absolute
    )

    figure, axis = plt.subplots(
        figsize=(11.5, 7.5),
    )

    scatter = axis.scatter(
        x_grid.ravel(),
        y_grid.ravel(),
        s=bubble_sizes.ravel(),
        c=matrix.ravel(),
        cmap="RdBu_r",
        vmin=-max_absolute,
        vmax=max_absolute,
        edgecolors="0.25",
        linewidths=0.5,
        alpha=0.88,
    )

    axis.set_xticks(
        np.arange(len(MODEL_ORDER))
    )
    axis.set_xticklabels(
        [model_label(value) for value in MODEL_ORDER],
        rotation=20,
        ha="right",
    )

    axis.set_yticks(
        np.arange(len(pivot))
    )
    axis.set_yticklabels(pivot.index)

    axis.set_xlabel("Modello")
    axis.set_ylabel("Token")
    axis.set_title(
        f"Segno e intensità dei contributi SHAP — {task_label(task_name)}"
    )

    for row_index in range(matrix.shape[0]):
        for column_index in range(matrix.shape[1]):
            value = matrix[
                row_index,
                column_index,
            ]

            if abs(value) >= 0.45 * max_absolute:
                axis.text(
                    column_index,
                    row_index,
                    f"{value:+.2f}",
                    ha="center",
                    va="center",
                    fontsize=7,
                )

    colorbar = figure.colorbar(
        scatter,
        ax=axis,
        shrink=0.88,
        pad=0.03,
    )
    colorbar.set_label(
        "Contributo SHAP firmato"
    )

    size_handles = []
    size_labels = []

    for fraction, label in [
        (0.25, "Basso"),
        (0.55, "Medio"),
        (1.00, "Alto"),
    ]:
        size_handles.append(
            axis.scatter(
                [],
                [],
                s=45 + 760 * fraction,
                color="0.78",
                edgecolor="0.25",
            )
        )
        size_labels.append(label)

    axis.legend(
        size_handles,
        size_labels,
        title="Importanza assoluta",
        bbox_to_anchor=(1.18, 0.42),
        loc="center left",
        frameon=True,
    )

    axis.grid(False)

    save_shap_task_figure(
        figure,
        task_name,
        "Bubble matrix con segno e intensità dei contributi",
    )


# -----------------------------------------------------------------------------
# STS-B: contribution swarm per modello.
# Mostra la distribuzione positiva/negativa e annota i token più forti.
# -----------------------------------------------------------------------------

def plot_shap_contribution_swarm(task_name):
    task_data = shap_summary_df[
        shap_summary_df[
            "task_name"
        ]
        == task_name
    ].copy()

    task_data = filter_view(task_data)

    if task_data.empty:
        return

    figure, axis = plt.subplots(
        figsize=(13, 7),
    )

    plotted_values = []

    for model_index, model_name in enumerate(MODEL_ORDER):
        model_data = (
            task_data[
                task_data[
                    "model_name"
                ]
                == model_name
            ]
            .groupby(
                "display_token",
                as_index=False,
            )
            .agg(
                shap_value=(
                    "shap_value",
                    "sum",
                ),
                absolute_shap_value=(
                    "absolute_shap_value",
                    "sum",
                ),
            )
            .sort_values(
                "absolute_shap_value",
                ascending=False,
            )
            .head(8)
            .reset_index(drop=True)
        )

        if model_data.empty:
            continue

        values = model_data[
            "shap_value"
        ].to_numpy()

        plotted_values.extend(values.tolist())

        offsets = np.linspace(
            -0.22,
            0.22,
            len(model_data),
        )

        y_positions = (
            model_index
            + offsets
        )

        max_model_importance = float(
            model_data[
                "absolute_shap_value"
            ].max()
        )

        if max_model_importance <= 0:
            max_model_importance = 1.0

        sizes = (
            55
            + 420
            * model_data[
                "absolute_shap_value"
            ].to_numpy()
            / max_model_importance
        )

        axis.scatter(
            values,
            y_positions,
            s=sizes,
            color=MODEL_COLORS[model_name],
            alpha=0.78,
            edgecolor="white",
            linewidth=0.6,
        )

        for row_index in range(min(3, len(model_data))):
            row = model_data.iloc[row_index]

            axis.annotate(
                row[
                    "display_token"
                ],
                (
                    row[
                        "shap_value"
                    ],
                    y_positions[row_index],
                ),
                xytext=(5, 4),
                textcoords="offset points",
                fontsize=8,
            )

    axis.axvline(
        0,
        linewidth=1.1,
        color="black",
    )

    axis.set_yticks(
        np.arange(len(MODEL_ORDER))
    )
    axis.set_yticklabels(
        [model_label(value) for value in MODEL_ORDER]
    )

    axis.set_xlabel("Contributo SHAP firmato")
    axis.set_ylabel("Modello")
    axis.set_title(
        f"Distribuzione dei principali contributi SHAP — {task_label(task_name)}"
    )

    if plotted_values:
        max_absolute = max(
            abs(value)
            for value in plotted_values
        )
        axis.set_xlim(
            -max_absolute * 1.28,
            max_absolute * 1.28,
        )

    save_shap_task_figure(
        figure,
        task_name,
        "Contribution swarm con annotazione dei token più forti",
    )


# -----------------------------------------------------------------------------
# Aggiorna la guida di lettura salvata nel chart_index.csv.
# -----------------------------------------------------------------------------

if "_chart_explanation_before_shap_redesign" not in globals():
    _chart_explanation_before_shap_redesign = chart_explanation


def chart_explanation(filename):
    explanation = _chart_explanation_before_shap_redesign(
        filename
    )

    if filename == "13_shap_reconstruction_error.png":
        explanation[
            "how_to_read"
        ] = (
            "Leggere la matrice per riga: ogni riga è un task e ogni colonna un modello. "
            "Celle chiare e valori vicini a zero indicano una ricostruzione più coerente; "
            "celle più scure segnalano gli abbinamenti da interpretare con maggiore cautela."
        )

    visual_guides = {
        task_name: (
            "Small multiples: ogni pannello rappresenta uno dei cinque modelli. "
            "I token sono allineati nello stesso ordine e la scala X è condivisa; "
            "le barre piene sostengono la label target, quelle tratteggiate la contrastano."
        )
        for task_name in TASK_ORDER
    }

    if filename.startswith(
        "14_shap_top_tokens_"
    ):
        for task_name, guide in visual_guides.items():
            if task_name in filename:
                explanation[
                    "how_to_read"
                ] = guide
                break

    return explanation


# La diagnostica è stata salvata prima della ridefinizione della guida.
# Aggiorna quindi la relativa riga già presente nell'indice.
for _chart_row in chart_index_rows:
    if _chart_row.get("filename") == "13_shap_reconstruction_error.png":
        _chart_row.update(
            chart_explanation(
                "13_shap_reconstruction_error.png"
            )
        )


def plot_shap_small_multiples(
    task_name,
    top_k=10,
):
    """
    Genera cinque pannelli affiancati, uno per modello, usando:
    - gli stessi token;
    - lo stesso ordine dei token;
    - la stessa scala dell'asse X;
    - barre tratteggiate e attenuate per i contributi negativi.
    """

    pivot = build_shap_task_pivot(
        task_name,
        top_k=top_k,
    )

    if pivot.empty:
        return

    max_absolute = float(
        pivot
        .abs()
        .to_numpy()
        .max()
    )

    if not np.isfinite(max_absolute) or max_absolute <= 0:
        max_absolute = 1.0

    x_limit = max_absolute * 1.18

    figure_height = max(
        6.5,
        0.58 * len(pivot) + 2.2,
    )

    figure, axes = plt.subplots(
        1,
        len(MODEL_ORDER),
        figsize=(19, figure_height),
        sharex=True,
        sharey=True,
        constrained_layout=True,
    )

    axes = np.atleast_1d(
        axes
    )

    token_positions = np.arange(
        len(pivot)
    )

    for model_index, model_name in enumerate(MODEL_ORDER):
        axis = axes[
            model_index
        ]

        values = pivot[
            model_name
        ].to_numpy(
            dtype=float
        )

        bars = axis.barh(
            token_positions,
            values,
            height=0.78,
            color=MODEL_COLORS[model_name],
            edgecolor=MODEL_COLORS[model_name],
            linewidth=0.7,
        )

        for bar, value in zip(
            bars,
            values,
        ):
            if value < 0:
                bar.set_alpha(0.38)
                bar.set_hatch("//")

            if abs(value) < max_absolute * 0.025:
                continue

            horizontal_alignment = (
                "left"
                if value >= 0
                else "right"
            )

            text_offset = (
                max_absolute * 0.025
                if value >= 0
                else -max_absolute * 0.025
            )

            axis.text(
                value + text_offset,
                bar.get_y() + bar.get_height() / 2,
                f"{value:+.2f}",
                va="center",
                ha=horizontal_alignment,
                fontsize=7.5,
                color="0.25",
            )

        axis.axvline(
            0,
            linewidth=1.15,
            color="black",
            zorder=3,
        )

        axis.set_xlim(
            -x_limit,
            x_limit,
        )

        axis.set_title(
            model_label(model_name),
            fontsize=11,
            pad=10,
        )

        axis.set_xlabel(
            "Valore SHAP"
        )

        axis.grid(
            axis="x",
            alpha=0.25,
        )

        if model_index == 0:
            axis.set_yticks(
                token_positions
            )
            axis.set_yticklabels(
                pivot.index
            )
            axis.set_ylabel(
                "Token"
            )
        else:
            axis.tick_params(
                axis="y",
                labelleft=False,
            )

    figure.suptitle(
        f"Contributi SHAP per modello — {task_label(task_name)}",
        fontsize=15,
        y=1.03,
    )

    figure.text(
        0.5,
        -0.015,
        (
            "Barre piene: contributo positivo verso la label target · "
            "Barre tratteggiate: contributo negativo"
        ),
        ha="center",
        fontsize=9,
        color="0.35",
    )

    save_shap_task_figure(
        figure,
        task_name,
        "Small multiples con un pannello per modello e scala condivisa",
    )

# -----------------------------------------------------------------------------
# Generazione uniforme dei contributi SHAP.
# Tutti i task usano small multiples: un pannello per ciascuno dei cinque
# modelli, token allineati e scala X condivisa.
# I nomi dei file restano invariati.
# -----------------------------------------------------------------------------

for task_name in TASK_ORDER:
    plot_shap_small_multiples(
        task_name,
        top_k=10,
    )


## 4. Attention

Output generati:

- sette grafici `21_attention_all_layers_mean_<task>.png`;
- sette grafici `21_attention_attention_rollout_<task>.png`;
- sette grafici `35_attention_layer_profile_<task>.png`;
- `46_adapter_vs_base_attention.png`.


In [ ]:
# CELLA 16 — Overview globale e top token Attention.

attention_prepared_df = prepare_tokens(
    attention_df,
    "token",
)


attention_summary_df = (
    attention_prepared_df[
        attention_prepared_df[
            "attention_type"
        ]
        .isin(
            ATTENTION_TYPES_TO_PLOT
        )
    ]
    .groupby(
        [
            "model_name",
            "task_name",
            "example_id",
            "attention_type",
            "display_token",
            "is_prompt_artifact",
        ],
        as_index=False,
    )
    .agg(
        attention_score=(
            "attention_score",
            "sum",
        ),
    )
)


ATTENTION_TYPE_DISPLAY = {
    "all_layers_mean": "Attention media",
    "attention_rollout": "Attention rollout",
}


# -----------------------------------------------------------------------------
# 1. Quadro globale dell'Attention, aggregato sui sette task.
#
# Il grafico è volutamente scorporato dai singoli test e sintetizza:
# - la quota di attention contenuta nei dieci token principali;
# - la quota di attention assorbita dagli artefatti strutturali del prompt.
# -----------------------------------------------------------------------------

def attention_group_metrics(group):
    scores = pd.to_numeric(
        group["attention_score"],
        errors="coerce",
    ).fillna(0.0)

    total_score = float(
        scores.sum()
    )

    if total_score <= 0:
        return pd.Series({
            "top10_concentration": np.nan,
            "prompt_artifact_share": np.nan,
        })

    top10_concentration = float(
        scores
        .nlargest(
            min(
                10,
                len(scores),
            )
        )
        .sum()
        / total_score
    )

    prompt_artifact_share = float(
        scores[
            group[
                "is_prompt_artifact"
            ]
            .to_numpy(
                dtype=bool
            )
        ]
        .sum()
        / total_score
    )

    return pd.Series({
        "top10_concentration": top10_concentration,
        "prompt_artifact_share": prompt_artifact_share,
    })


attention_global_metrics_df = (
    attention_summary_df
    .groupby(
        [
            "model_name",
            "task_name",
            "example_id",
            "attention_type",
        ],
        group_keys=False,
    )
    .apply(
        attention_group_metrics,
        include_groups=False,
    )
    .reset_index()
)


attention_global_overview_df = (
    attention_global_metrics_df
    .groupby(
        [
            "model_name",
            "attention_type",
        ],
        as_index=False,
    )
    .agg(
        top10_concentration=(
            "top10_concentration",
            "mean",
        ),
        prompt_artifact_share=(
            "prompt_artifact_share",
            "mean",
        ),
    )
)


overview_figure, overview_axes = plt.subplots(
    1,
    2,
    figsize=(
        15,
        6.5,
    ),
    sharey=False,
)


overview_metrics = [
    (
        "top10_concentration",
        "Concentrazione sui 10 token principali",
        "Quota dell'attention totale",
    ),
    (
        "prompt_artifact_share",
        "Attention sugli artefatti del prompt",
        "Quota dell'attention totale",
    ),
]


attention_type_colors = {
    attention_type: _default_model_colors[
        index % len(_default_model_colors)
    ]
    for index, attention_type in enumerate(
        ATTENTION_TYPES_TO_PLOT
    )
}


model_positions = np.arange(
    len(MODEL_ORDER)
)

bar_width = 0.36


for axis, (
    metric_column,
    panel_title,
    y_label,
) in zip(
    overview_axes,
    overview_metrics,
):
    for attention_index, attention_type in enumerate(
        ATTENTION_TYPES_TO_PLOT
    ):
        metric_values = (
            attention_global_overview_df[
                attention_global_overview_df[
                    "attention_type"
                ]
                == attention_type
            ]
            .set_index(
                "model_name"
            )[
                metric_column
            ]
            .reindex(
                MODEL_ORDER
            )
            .to_numpy(
                dtype=float
            )
        )

        offset = (
            attention_index
            - (
                len(
                    ATTENTION_TYPES_TO_PLOT
                )
                - 1
            )
            / 2
        ) * bar_width

        bars = axis.bar(
            model_positions + offset,
            metric_values,
            width=bar_width,
            label=ATTENTION_TYPE_DISPLAY.get(
                attention_type,
                attention_type,
            ),
            color=attention_type_colors[
                attention_type
            ],
            alpha=0.88,
        )

        for bar, value in zip(
            bars,
            metric_values,
        ):
            if not np.isfinite(value):
                continue

            axis.text(
                bar.get_x()
                + bar.get_width()
                / 2,
                value + 0.018,
                f"{value:.0%}",
                ha="center",
                va="bottom",
                fontsize=8,
            )

    axis.set_title(
        panel_title
    )

    axis.set_xticks(
        model_positions
    )

    axis.set_xticklabels(
        [
            model_label(
                model_name
            )
            for model_name in MODEL_ORDER
        ],
        rotation=20,
        ha="right",
    )

    axis.set_ylabel(
        y_label
    )

    axis.set_ylim(
        0,
        1.08,
    )

    axis.yaxis.set_major_formatter(
        plt.FuncFormatter(
            lambda value, _: f"{value:.0%}"
        )
    )

    axis.grid(
        axis="y",
        alpha=0.25,
    )


overview_axes[1].legend(
    title="Metodo",
    loc="upper right",
)


overview_figure.suptitle(
    "Profilo globale dell'Attention media — aggregato sui sette task",
    fontsize=15,
    y=1.02,
)


overview_figure.text(
    0.5,
    -0.025,
    (
        "Sinistra: quanto l'attention è concentrata sui token dominanti · "
        "Destra: quanto peso viene assorbito dalla struttura del prompt"
    ),
    ha="center",
    fontsize=9,
    color="0.35",
)


save_figure(
    overview_figure,
    "20_attention_global_overview.png",
    "attention",
    (
        "Profilo globale dell'Attention aggregato sui sette task: "
        "concentrazione sui top token e quota sugli artefatti del prompt."
    ),
)


# -----------------------------------------------------------------------------
# 2. Small multiples per task e tipo di Attention.
# -----------------------------------------------------------------------------

def build_attention_task_pivot(
    task_name,
    attention_type,
    top_k=10,
):
    task_data = attention_summary_df[
        (
            attention_summary_df[
                "task_name"
            ]
            == task_name
        )
        & (
            attention_summary_df[
                "attention_type"
            ]
            == attention_type
        )
    ].copy()

    task_data = filter_view(
        task_data
    )

    if task_data.empty:
        return pd.DataFrame()

    token_ranking = (
        task_data
        .groupby(
            "display_token",
            as_index=False,
        )[
            "attention_score"
        ]
        .mean()
        .sort_values(
            "attention_score",
            ascending=False,
        )
        .head(
            top_k
        )
    )

    selected_tokens = (
        token_ranking[
            "display_token"
        ]
        .tolist()
    )

    pivot = (
        task_data[
            task_data[
                "display_token"
            ]
            .isin(
                selected_tokens
            )
        ]
        .pivot_table(
            index="display_token",
            columns="model_name",
            values="attention_score",
            aggfunc="sum",
            fill_value=0.0,
        )
        .reindex(
            columns=MODEL_ORDER,
            fill_value=0.0,
        )
    )

    ordering = (
        pivot
        .mean(
            axis=1
        )
        .sort_values(
            ascending=True
        )
        .index
    )

    return pivot.loc[
        ordering
    ]


def format_attention_value(
    value,
    maximum,
):
    if maximum < 0.01:
        return f"{value:.4f}"

    if maximum < 0.1:
        return f"{value:.3f}"

    return f"{value:.2f}"


def plot_attention_small_multiples(
    task_name,
    attention_type,
    top_k=10,
):
    pivot = build_attention_task_pivot(
        task_name,
        attention_type,
        top_k=top_k,
    )

    if pivot.empty:
        return

    maximum = float(
        pivot
        .to_numpy(
            dtype=float
        )
        .max()
    )

    if not np.isfinite(maximum) or maximum <= 0:
        maximum = 1.0

    x_limit = maximum * 1.22

    figure_height = max(
        6.5,
        0.58 * len(pivot) + 2.2,
    )

    figure, axes = plt.subplots(
        1,
        len(MODEL_ORDER),
        figsize=(
            19,
            figure_height,
        ),
        sharex=True,
        sharey=True,
    )

    axes = np.atleast_1d(
        axes
    )

    token_positions = np.arange(
        len(pivot)
    )

    for model_index, model_name in enumerate(
        MODEL_ORDER
    ):
        axis = axes[
            model_index
        ]

        values = pivot[
            model_name
        ].to_numpy(
            dtype=float
        )

        bars = axis.barh(
            token_positions,
            values,
            height=0.78,
            color=MODEL_COLORS[
                model_name
            ],
            edgecolor=MODEL_COLORS[
                model_name
            ],
            linewidth=0.7,
            alpha=0.9,
        )

        for bar, value in zip(
            bars,
            values,
        ):
            if value <= maximum * 0.02:
                continue

            axis.text(
                value + maximum * 0.025,
                bar.get_y()
                + bar.get_height()
                / 2,
                format_attention_value(
                    value,
                    maximum,
                ),
                va="center",
                ha="left",
                fontsize=7.5,
                color="0.25",
            )

        axis.set_xlim(
            0,
            x_limit,
        )

        axis.set_title(
            model_label(
                model_name
            ),
            fontsize=11,
            pad=10,
        )

        axis.set_xlabel(
            "Attention score"
        )

        axis.grid(
            axis="x",
            alpha=0.25,
        )

        if model_index == 0:
            axis.set_yticks(
                token_positions
            )

            axis.set_yticklabels(
                pivot.index
            )

            axis.set_ylabel(
                "Token"
            )
        else:
            axis.tick_params(
                axis="y",
                labelleft=False,
            )

    figure.suptitle(
        (
            f"{ATTENTION_TYPE_DISPLAY.get(attention_type, attention_type)} "
            f"per modello — {task_label(task_name)}"
        ),
        fontsize=15,
        y=0.995,
    )

    figure.subplots_adjust(
        top=0.88,
        bottom=0.13,
        left=0.08,
        right=0.98,
        wspace=0.08,
    )

    save_figure(
        figure,
        (
            "21_attention_"
            f"{attention_type}_"
            f"{task_name}.png"
        ),
        "attention",
        (
            f"{ATTENTION_TYPE_DISPLAY.get(attention_type, attention_type)} "
            f"per {task_label(task_name)}, con un pannello per modello."
        ),
    )


for attention_type in ATTENTION_TYPES_TO_PLOT:
    for task_name in TASK_ORDER:
        plot_attention_small_multiples(
            task_name,
            attention_type,
            top_k=10,
        )


## 5. Influenza della struttura del prompt

Output generati:

- `42_prompt_artifact_share_lime.png`;
- `42_prompt_artifact_share_shap.png`;
- `42_prompt_artifact_share_attention_mean.png`.


## 6. Confronto finale

Output generati:

- `45_cross_method_overlap.png`;
- `46_adapter_vs_base_lime.png`;
- `46_adapter_vs_base_shap.png`;
- `46_adapter_vs_base_attention.png`.


## 7. Salvataggio finale

Vengono salvati soltanto:

- i 45 PNG previsti dalla scaletta della presentazione;
- `chart_index.csv`;
- `explainability_visual_summary.csv`.


In [ ]:
# CELLA FINALE — Indice dei 25 grafici e guida delle slide.

chart_index_df = (
    pd.DataFrame(
        chart_index_rows
    )
    .sort_values(
        [
            "slide_number",
            "filename",
        ],
        na_position="last",
    )
    .reset_index(
        drop=True
    )
)


chart_index_df.to_csv(
    OUTPUT_DIR / "chart_index.csv",
    index=False,
    encoding="utf-8-sig",
)


final_summary_df = (
    prediction_summary_df
    .merge(
        lime_diagnostics_df[
            [
                "model_name",
                "task_name",
                "example_id",
                "lime_local_score",
                "local_approximation_error",
            ]
        ],
        on=[
            "model_name",
            "task_name",
            "example_id",
        ],
        how="left",
        validate="one_to_one",
    )
    .merge(
        shap_diagnostics_df[
            [
                "model_name",
                "task_name",
                "example_id",
                "reconstruction_error",
            ]
        ],
        on=[
            "model_name",
            "task_name",
            "example_id",
        ],
        how="left",
        validate="one_to_one",
    )
)


final_summary_df.to_csv(
    OUTPUT_DIR / "explainability_visual_summary.csv",
    index=False,
    encoding="utf-8-sig",
)


expected_chart_count = (
    2     # diagnostica LIME
    + 7   # contributi LIME per task
    + 1   # diagnostica SHAP
    + 7   # contributi SHAP per task
    + 1   # overview globale Attention media
    + 7   # Attention media per task
)


if len(chart_index_df) != expected_chart_count:
    raise RuntimeError(
        "Numero di grafici inatteso: "
        f"{len(chart_index_df)} invece di {expected_chart_count}. "
        "La cartella dovrebbe contenere esclusivamente i 25 grafici approvati."
    )


presentation_slide_guide_df = chart_index_df[
    [
        "slide_number",
        "slide_title",
        "filename",
        "what_it_shows",
        "why_it_matters",
        "how_to_read",
        "caution",
    ]
].copy()


presentation_slide_guide_df.to_csv(
    OUTPUT_DIR / "presentation_slide_guide.csv",
    index=False,
    encoding="utf-8-sig",
)


print(
    "Grafici prodotti:",
    len(chart_index_df),
)

print(
    "Cartella figure:",
    FIGURES_DIR,
)

print(
    "Guida slide:",
    OUTPUT_DIR / "presentation_slide_guide.csv",
)

display(
    presentation_slide_guide_df
)


# Output definitivo per le slide

Il notebook genera esclusivamente **25 grafici**, uno per slide:

- **9 grafici LIME**: due diagnostiche e sette task;
- **8 grafici SHAP**: una diagnostica e sette task;
- **8 grafici Attention**: un overview globale e sette grafici di Attention media.

Sono esclusi Attention rollout, profili per layer, confronti con il base, accordo tra metodi e grafici separati sugli artefatti del prompt.

La cartella di output contiene anche `presentation_slide_guide.csv`, con titolo, significato, rilevanza e cautela interpretativa per ogni slide.
